# 03 – Merge Pre-1997 Legacy Decisions into EC Antitrust Master

## Ziel

Dieses Notebook integriert die Legacy Formal Decisions (pre-1997) in den bestehenden EC-Antitrust-Master.
Das Ergebnis ist eine aktualisierte `ec_antitrust_master.csv` mit genau diesen Spalten:
`ec_case_number`, `case_title`, `date`, `type`, `celex_no`


## 1. Imports


In [1]:
import pandas as pd
from pathlib import Path

print("Imports OK")


Imports OK


## 2. Pfade


In [2]:
MASTER_PATH = Path("data/processed/ec_antitrust_master.csv")
LEGACY_PATH = Path("data/processed/ec_legacy_formal_decisions.csv")
OUTPUT_PATH = Path("data/processed/ec_antitrust_master.csv")

FINAL_COLUMNS = ["ec_case_number", "case_title", "date", "type", "celex_no"]

print(f"Master : {MASTER_PATH}")
print(f"Legacy : {LEGACY_PATH}")
print(f"Output : {OUTPUT_PATH}")


Master : data\processed\ec_antitrust_master.csv
Legacy : data\processed\ec_legacy_formal_decisions.csv
Output : data\processed\ec_antitrust_master.csv


## 3. Master laden


In [3]:
df_master = pd.read_csv(MASTER_PATH, dtype=str)
print(f"Master geladen: {len(df_master)} Zeilen, Spalten: {df_master.columns.tolist()}")
df_master.head(3)


Master geladen: 748 Zeilen, Spalten: ['ec_case_number', 'case_title', 'date', 'type']


,ec_case_number,case_title,date,type
0,AT.35803,IPEX Consortium,NaN,AtStandardATCCase
1,AT.34950,ECO EMBALLAGES,2001-06-15,AtStandardATCCase
2,AT.39172,Electricity sector inquiry,2007-01-10,AtSectorInquiryCase


## 4. Legacy laden


In [4]:
df_legacy = pd.read_csv(LEGACY_PATH, dtype=str)
print(f"Legacy geladen: {len(df_legacy)} Zeilen, Spalten: {df_legacy.columns.tolist()}")
df_legacy.head(3)


Legacy geladen: 437 Zeilen, Spalten: ['year', 'source_url', 'title', 'date', 'decision_type', 'official_journal', 'case_numbers', 'celex_no', 'document_url']


,year,source_url,title,date,decision_type,official_journal,case_numbers,celex_no,document_url
0,1964,https://ec.europa.eu/competition/antitrust/clo...,Deca,22.10.1964,Negative clearance Art.81(1) [ex 85(1)],L - 31/10/1964 Page : 2761,IV/71,31964D0599,https://eur-lex.europa.eu/legal-content/EN/TXT...
1,1964,https://ec.europa.eu/competition/antitrust/clo...,Grundig-Consten,23.09.1964,Infringement Art.81 [ex 85],L - 20/10/1964 Page : 2545,IV/3344; IV/4,31964D0566,https://eur-lex.europa.eu/legal-content/EN/TXT...
2,1964,https://ec.europa.eu/competition/antitrust/clo...,Nicholas Freres + Vitapro,30.07.1964,Negative clearance Art.81(1) [ex 85(1)],L - 26/08/1964 Page : 2287,IV/95,31964D0502,https://eur-lex.europa.eu/legal-content/EN/TXT...


## 5. Spalten auf Zielschema mappen


In [5]:
# Master: Spalten bereits passend (ec_case_number, case_title, date, type)
# celex_no fehlt im Master -> leer lassen
df_master_mapped = df_master.reindex(columns=FINAL_COLUMNS)

print(f"Master gemappt: {df_master_mapped.shape}")
df_master_mapped.head(3)


Master gemappt: (748, 5)


,ec_case_number,case_title,date,type,celex_no
0,AT.35803,IPEX Consortium,NaN,AtStandardATCCase,NaN
1,AT.34950,ECO EMBALLAGES,2001-06-15,AtStandardATCCase,NaN
2,AT.39172,Electricity sector inquiry,2007-01-10,AtSectorInquiryCase,NaN


In [6]:
# Legacy: Spalten umbenennen und auf Zielschema bringen
# Vorhandene Spalten in ec_legacy_formal_decisions.csv:
#   year, title, date, decision_type, official_journal, celex_no, case_numbers, source_url, document_url
df_legacy_mapped = df_legacy.rename(columns={
    "case_numbers": "ec_case_number",
    "title":        "case_title",
    "date":         "date",
    "celex_no":     "celex_no",
})

# type als konstanter Wert
df_legacy_mapped["type"] = "formal_decision"

df_legacy_mapped = df_legacy_mapped.reindex(columns=FINAL_COLUMNS)

print(f"Legacy gemappt: {df_legacy_mapped.shape}")
df_legacy_mapped.head(3)


Legacy gemappt: (437, 5)


,ec_case_number,case_title,date,type,celex_no
0,IV/71,Deca,22.10.1964,formal_decision,31964D0599
1,IV/3344; IV/4,Grundig-Consten,23.09.1964,formal_decision,31964D0566
2,IV/95,Nicholas Freres + Vitapro,30.07.1964,formal_decision,31964D0502


## 6. Zusammenführen


In [7]:
df_combined = pd.concat([df_master_mapped, df_legacy_mapped], ignore_index=True)
print(f"Kombiniert: {len(df_combined)} Zeilen")


Kombiniert: 1185 Zeilen


## 7. Finale Spaltenreihenfolge


In [8]:
df_final = df_combined[FINAL_COLUMNS]
print(f"Finale Spalten: {df_final.columns.tolist()}")
print(f"Finale Shape  : {df_final.shape}")
df_final.head(5)


Finale Spalten: ['ec_case_number', 'case_title', 'date', 'type', 'celex_no']
Finale Shape  : (1185, 5)


,ec_case_number,case_title,date,type,celex_no
0,AT.35803,IPEX Consortium,NaN,AtStandardATCCase,NaN
1,AT.34950,ECO EMBALLAGES,2001-06-15,AtStandardATCCase,NaN
2,AT.39172,Electricity sector inquiry,2007-01-10,AtSectorInquiryCase,NaN
3,AT.39294,Microsoft (ECIS complaint),NaN,AtStandardATCCase,NaN
4,AT.39173,Gas sector inquiry,2007-01-10,AtSectorInquiryCase,NaN


## 8. ec_antitrust_master.csv speichern


In [9]:
df_final.to_csv(OUTPUT_PATH, index=False, encoding="utf-8")
print(f"✅ Gespeichert: {OUTPUT_PATH.resolve()}")
print(f"   {len(df_final)} Zeilen, {len(df_final.columns)} Spalten")


✅ Gespeichert: C:\vsCodeWorkspace\masterZHAW\masterarbeit\data_analysis_master\data\processed\ec_antitrust_master.csv
   1185 Zeilen, 5 Spalten
